# Anomali / spam tak berlabel

Model mengurutkan kandidat tidak biasa; bukan alasan menghapusnya otomatis. Baca contoh per panjang/bahasa/sumber.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DATA, TEXT = "train.csv", "text"
POOL = ["zxcv qwer asdf lorem ipsum", "beli slot gacor promo klik sekarang", "aaaa aaaa aaaa aaaa"]   # untuk uji injeksi


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
df = read_any(DATA)
print(df.shape)


## Preprocessing

Teks dipertahankan agar keanehan tetap terlihat.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
docs = preprocess_corpus(df[TEXT], "tfidf")


## Skor anomali

Gabungan IForest, kNN, dan rekonstruksi.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import FunctionTransformer, Normalizer

def _tokens(t):
    return re.findall(r"\w+", str(t).lower())

def _meta(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    return sparse.csr_matrix(pd.DataFrame({"len": np.log1p(n), "words": np.log1p(s.str.split().str.len()), "upper": s.str.count(r"[A-Z]") / n, "digit": s.str.count(r"\d") / n,
                                           "punct": s.str.count(r"[^\w\s]") / n, "excl": s.str.count("!").clip(upper=5), "url": s.str.count(r"https?://|URL"), "mention": s.str.count(r"@\w+|USER")}).to_numpy("float32"))

class RawText(BaseEstimator, TransformerMixin):
    """Teks mentah untuk model neural."""
    def fit(self, X, y=None): return self
    def transform(self, X): return list(X)

class GensimVec(BaseEstimator, TransformerMixin):
    """kind: w2v, fasttext, doc2vec. idf=True: rata-rata berbobot."""
    def __init__(self, kind="w2v", dim=200, window=5, min_count=2, epochs=5, idf=False, workers=4, seed=42):
        self.kind, self.dim, self.window, self.min_count, self.epochs, self.idf, self.workers, self.seed = kind, dim, window, min_count, epochs, idf, workers, seed
    def fit(self, X, y=None):
        from gensim.models import FastText, Word2Vec
        from gensim.models.doc2vec import Doc2Vec, TaggedDocument
        toks = [_tokens(t) for t in X]
        kw = dict(vector_size=self.dim, min_count=self.min_count, epochs=self.epochs, workers=self.workers, seed=self.seed)
        if self.kind == "doc2vec":
            self.m_ = Doc2Vec([TaggedDocument(t, [i]) for i, t in enumerate(toks)], dm=0, **kw)
        else:
            self.m_ = (FastText if self.kind == "fasttext" else Word2Vec)(toks, window=self.window, **kw).wv
        self.w_ = None
        if self.idf:
            t = TfidfVectorizer(token_pattern=r"\w+").fit(X)
            self.w_ = dict(zip(t.get_feature_names_out(), t.idf_))
        return self
    def transform(self, X):
        out = np.zeros((len(X), self.dim), dtype="float32")
        for i, t in enumerate(X):
            w = _tokens(t)
            if self.kind == "doc2vec":
                out[i] = self.m_.infer_vector(w)
                continue
            w = w if self.kind == "fasttext" else [x for x in w if x in self.m_]
            if w:
                a = np.array([self.w_.get(x, 1.0) for x in w])[:, None] if self.w_ else np.ones((len(w), 1))
                out[i] = (self.m_[w] * a).sum(0) / a.sum()
        return out

class EmbedVec(BaseEstimator, TransformerMixin):
    """Embedding SentenceTransformer dengan cache per teks agar tiap fold tidak encode ulang."""
    _cache = {}
    def __init__(self, model_path=None, prefix="", batch=128):
        self.model_path, self.prefix, self.batch = model_path, prefix, batch
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = EmbedVec._cache.setdefault(str(self.model_path), {})
        miss = [t for t in dict.fromkeys(map(str, X)) if t not in c]
        if miss:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer(str(self.model_path), local_files_only=True)
            for t, v in zip(miss, m.encode([self.prefix + t for t in miss], batch_size=self.batch, normalize_embeddings=True, show_progress_bar=False)):
                c[t] = v.astype("float32")
        return np.vstack([c[str(t)] for t in X])

def make_vectorizer(kind="tfidf_union", **kw):
    """Semua hasilnya punya fit_transform dan transform."""
    word = dict(ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True, dtype="float32", token_pattern=r"(?u)\b\w+\b")
    char = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=500_000, sublinear_tf=True, dtype="float32")
    if kind == "tfidf_word": return TfidfVectorizer(**{**word, **kw})
    if kind == "tfidf_char": return TfidfVectorizer(**{**char, **kw})
    if kind == "tfidf_union": return make_union(TfidfVectorizer(**word), TfidfVectorizer(**char))
    if kind == "count": return CountVectorizer(**{**dict(ngram_range=(1, 1), min_df=2, max_df=.95), **kw})
    if kind == "binary": return CountVectorizer(**{**dict(ngram_range=(1, 2), min_df=2, binary=True), **kw})
    if kind == "hashing": return HashingVectorizer(**{**dict(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), dtype="float32"), **kw})
    if kind == "lsa": return make_pipeline(TfidfVectorizer(**word), TruncatedSVD(kw.pop("k", 256), random_state=42), Normalizer(copy=False))
    if kind == "meta": return FunctionTransformer(_meta, validate=False)
    if kind == "tfidf_meta": return make_union(TfidfVectorizer(**word), FunctionTransformer(_meta, validate=False))
    if kind in ("w2v", "fasttext", "doc2vec"): return GensimVec(kind, **kw)
    if kind == "w2v_idf": return GensimVec("w2v", idf=True, **kw)
    if kind == "embed": return EmbedVec(**kw)
    if kind == "raw": return RawText()
    raise ValueError(f"vectorizer tidak dikenal: {kind}")

import numpy as np
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

def iforest_score(Z, seed=42):
    return -IsolationForest(n_estimators=200, random_state=seed, n_jobs=4).fit(Z).score_samples(Z)

def knn_score(Z, k=10):
    d, _ = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(Z).kneighbors(Z)
    return d[:, 1:].mean(1)

def lof_score(Z, k=20):
    m = LocalOutlierFactor(n_neighbors=k, metric="cosine").fit(Z)
    return -m.negative_outlier_factor_

def recon_score(X, k=100, seed=42):
    """Sisa rekonstruksi: |x|^2 - |proyeksi|^2. Aman untuk sparse besar, tanpa membuat matriks padat."""
    Zk = TruncatedSVD(k, random_state=seed).fit_transform(X)
    sq = np.asarray(X.multiply(X).sum(1)).ravel() if hasattr(X, "multiply") else (np.asarray(X) ** 2).sum(1)
    return np.maximum(sq - (Zk ** 2).sum(1), 0)

def rank_avg(*scores):
    return np.mean([rankdata(s) / len(s) for s in scores], 0)

def inject_eval(texts, make_scores, pool, share=.02, seed=42):
    """make_scores(list teks) -> skor. pool: teks asing yang disisipkan. Kembalikan ROC-AUC dan recall@share."""
    rng = np.random.default_rng(seed)
    n = max(1, int(len(texts) * share))
    ins = list(rng.choice(pool, n))
    s = np.asarray(make_scores(list(texts) + ins))
    lab = np.r_[np.zeros(len(texts)), np.ones(n)]
    top = np.argsort(-s)[:n]
    return {"auc": round(roc_auc_score(lab, s), 4), "recall@share": round(lab[top].mean(), 4)}

# ===== jalankan =====
def scores_of(t):
    X = make_vectorizer("tfidf_word", min_df=1).fit_transform(t)
    Z = make_vectorizer("lsa", k=32).fit_transform(t)
    return rank_avg(iforest_score(Z), knn_score(Z, 10), recon_score(X, 32))

df["anomaly"] = scores_of(docs)


## Uji injeksi

Seed uji berbeda dari tuning. Periksa korelasi dengan panjang.

In [ ]:
print(inject_eval(docs, scores_of, POOL, seed=7))
print("korelasi panjang", round(df["anomaly"].corr(df[TEXT].fillna("").str.len()), 3))
print(df.sort_values("anomaly", ascending=False)[TEXT].head(10).to_string())


## Output

Skor per baris.

In [ ]:
df.to_csv("submission.csv", index=False)
